# 12 — Inside the MLP: Feature Detection and Neurons

**Description:** Open the Transformer MLP, follow embeddings through its up-projection and activation, and inspect which hidden neurons respond to different input features.
**Level:** Beginner
**Tags:** Language Models, Transformers, MLP, Neurons, GELU, Feature Detection

Notebook 11 placed a position-wise MLP after attention. Attention moves information between token positions; the MLP transforms the information available at each position. This notebook opens the first half of that MLP:

$$x \rightarrow xW_{up}+b_{up} \rightarrow \text{activation} \rightarrow h$$

By the end, you will be able to explain the up-projection, interpret a pre-activation as a feature score, compare ReLU and GELU, and inspect hidden-neuron activations. We postpone the down-projection and residual update until Notebook 13.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. The MLP acts on one position at a time

After attention, each token position holds a contextual vector. The same MLP is applied independently to every position—there is no mixing across the token axis inside this sublayer.

We use four interpretable input coordinates: `person`, `sport`, `place`, and `action`. These are teaching features, not claims about literal coordinates in a trained model.

In [ ]:
feature_names = ["person", "sport", "place", "action"]
tokens = ["athlete", "stadium", "running", "teacher"]
X = np.array([
    [1.0, 0.8, 0.0, 0.2],
    [0.0, 0.7, 1.0, 0.0],
    [0.1, 0.6, 0.0, 1.0],
    [1.0, 0.0, 0.2, 0.2],
])
print("X shape:", X.shape)
for token, vector in zip(tokens, X):
    print(f"{token:>8}: {dict(zip(feature_names, vector))}")

## 2. Up-projection creates a wider hidden space

The first linear layer maps $d_{model}$ input features to $d_{mlp}$ hidden units:

$$P=XW_{up}+b_{up}$$

Transformer MLPs commonly use $d_{mlp}>d_{model}$. The wider space gives the layer many learned detectors to work with.

In [ ]:
W_up = np.array([
    [ 1.0,  0.0,  0.8, -0.5,  0.2,  0.7],
    [ 1.0,  0.8,  0.0,  0.5, -0.3,  0.1],
    [ 0.0,  1.0,  0.0,  0.0,  0.9, -0.2],
    [ 0.0,  0.0,  1.0,  0.7,  0.1,  0.6],
])
b_up = np.array([-1.0, -0.7, -0.6, -0.3, -0.4, -0.5])
pre = X @ W_up + b_up

print("X:    ", X.shape)
print("W_up: ", W_up.shape)
print("b_up: ", b_up.shape)
print("pre:  ", pre.shape)

The columns of $W_{up}$ define six input directions. A hidden unit's pre-activation is large when the input aligns with its column strongly enough to overcome the bias.

## 3. One neuron as a detector

Neuron 0 adds the `person` and `sport` coordinates, then subtracts a threshold of 1. Its pre-activation is:

$$p_0 = x_{person}+x_{sport}-1$$

In [ ]:
neuron = 0
manual = X[0] @ W_up[:, neuron] + b_up[neuron]
print("athlete input:       ", X[0])
print("neuron 0 direction: ", W_up[:, neuron])
print("manual score:       ", manual)
print("matrix result:      ", pre[0, neuron])
assert np.isclose(manual, pre[0, neuron])

This motivates the phrase **feature detector**: the neuron responds to a pattern across input coordinates. But it is only a toy interpretation. In real networks, a neuron may respond to several unrelated patterns, and a feature may be distributed across many neurons.

## 4. A nonlinearity gates the scores

Without an activation, two linear layers collapse into one linear transformation. A nonlinear activation lets the MLP respond differently in different regions of input space.

ReLU clips negative inputs to zero. GELU smoothly scales inputs, allowing small negative outputs.

In [ ]:
def relu(x):
    return np.maximum(0.0, x)

def gelu(x):
    # tanh approximation used in many implementations
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

grid = np.linspace(-4, 4, 500)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid, relu(grid), label="ReLU", linewidth=2)
ax.plot(grid, gelu(grid), label="GELU", linewidth=2)
ax.plot(grid, grid, "--", color="gray", alpha=0.5, label="identity")
ax.axhline(0, color="black", linewidth=0.7)
ax.set(xlabel="pre-activation", ylabel="activation", title="ReLU and GELU gate feature scores")
ax.legend()
plt.show()

### Compare a few values

In [ ]:
test_values = np.array([-3.0, -1.0, -0.2, 0.0, 0.2, 1.0, 3.0])
print(f"{'input':>8} {'ReLU':>8} {'GELU':>8}")
for value, r, g in zip(test_values, relu(test_values), gelu(test_values)):
    print(f"{value:8.2f} {r:8.3f} {g:8.3f}")

## 5. Inspect the whole hidden layer

Apply GELU to every pre-activation. Each row is one token position; each column is one hidden neuron.

In [ ]:
hidden = gelu(pre)
neuron_names = ["person+sport", "sport+place", "person+action", "action-ish", "place-ish", "person+action mix"]

fig, ax = plt.subplots(figsize=(9, 4))
image = ax.imshow(hidden, cmap="magma", aspect="auto")
ax.set_xticks(range(len(neuron_names)), neuron_names, rotation=30, ha="right")
ax.set_yticks(range(len(tokens)), tokens)
ax.set(xlabel="hidden neuron", ylabel="input token", title="GELU activations after the up-projection")
for i in range(len(tokens)):
    for j in range(len(neuron_names)):
        ax.text(j, i, f"{hidden[i, j]:.2f}", ha="center", va="center", color="white" if hidden[i, j] > 0.45 else "black")
fig.colorbar(image, ax=ax, label="activation")
plt.show()

### Your turn: probe a new input

Create a vector for `coach`, `museum`, or `swimming`. Predict which hidden neurons will be strongest before computing `gelu(new_x @ W_up + b_up)`. A probe tests a hypothesis about what a neuron detects; it does not prove a universal interpretation.

## 6. Why width helps

Each hidden neuron can test a different direction and threshold. Increasing $d_{mlp}$ creates more opportunities for conditional computation without changing the model-stream width. Only the activated pattern continues to the second MLP layer.

In [ ]:
strongest = np.argmax(hidden, axis=1)
for token, index in zip(tokens, strongest):
    print(f"{token:>8} most activates neuron {index}: {neuron_names[index]}")

## 7. The same operation in PyTorch

In [ ]:
import torch
from torch import nn

layer = nn.Linear(4, 6)
with torch.no_grad():
    layer.weight.copy_(torch.tensor(W_up.T, dtype=torch.float32))
    layer.bias.copy_(torch.tensor(b_up, dtype=torch.float32))

torch_hidden = nn.functional.gelu(layer(torch.tensor(X, dtype=torch.float32)))
print(torch_hidden)
assert np.allclose(torch_hidden.detach().numpy(), hidden, atol=5e-4)

## 8. Challenges

1. Design a neuron that activates only when both `place` and `action` are strong.
2. Remove the bias. How does the detector threshold change?
3. Replace GELU with ReLU and compare the heatmaps.
4. Show algebraically why two linear layers without an activation are equivalent to one.
5. Apply the same MLP to a tensor shaped `(batch, tokens, d_model)` in PyTorch.

## Takeaways

- A Transformer MLP processes each token position independently.
- The up-projection maps the model stream into a wider hidden space.
- Each pre-activation is a dot product plus a bias and can act like a learned feature score.
- ReLU and GELU introduce nonlinear gating, preventing the two MLP layers from collapsing into one linear map.
- Hidden neurons are useful units to inspect, but one neuron should not automatically be equated with one concept.
- Notebook 13 will project these activations down and write an update into the residual stream.